# Avaliação do modelo fine-tuned

Compara o **Llama 3.2 3B base** com o **fine-tuned (base + LoRA adapter)** no conjunto `sft_test.jsonl`.


In [ ]:
!pip install -q -U transformers peft accelerate bitsandbytes datasets rouge_score python-dotenv

In [ ]:
import os, json, re, torch
from pathlib import Path
from collections import defaultdict
from huggingface_hub import login
from google.colab import drive
from dotenv import load_dotenv
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from rouge_score import rouge_scorer

In [ ]:
drive.mount('/content/drive', force_remount=True)
DRIVE_BASE = '/content/drive/MyDrive/AssistenteHospitalar'
load_dotenv(f'{DRIVE_BASE}/.env')
login(token=os.getenv('HF_TOKEN'))
print('Autenticado.')

In [ ]:
MODEL_ID    = 'meta-llama/Llama-3.2-3B-Instruct'
FT_BASE     = Path(f'{DRIVE_BASE}/files/finetune')
# Selecione a run mais recente automaticamente
runs = sorted(FT_BASE.glob('llama32-3b-saude-mulher_*'))
if not runs:
    raise FileNotFoundError('Nenhuma run de fine-tuning encontrada em ' + str(FT_BASE))
ADAPTER_DIR = runs[-1] / 'adapter_final'
print('Adapter:', ADAPTER_DIR)

TEST_FILE = Path(f'{DRIVE_BASE}/files/sft/sft_test.jsonl')
OUT_REPORT = ADAPTER_DIR.parent / 'eval_report.json'
OUT_SIDEBYSIDE = ADAPTER_DIR.parent / 'eval_sidebyside.md'
print('Test :', TEST_FILE)

In [ ]:
ds_test = load_dataset('json', data_files=str(TEST_FILE))['train']
print(f'Test set: {len(ds_test)} exemplos')
print('Categorias:', set(ds_test['category']))

In [ ]:
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Carregando base...')
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map='auto', torch_dtype=torch.bfloat16,
)
base.eval()

print('Carregando fine-tuned (base + adapter)...')
ft = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map='auto', torch_dtype=torch.bfloat16,
)
ft = PeftModel.from_pretrained(ft, str(ADAPTER_DIR))
ft.eval()
print('Modelos prontos.')

In [ ]:
def gerar(model, messages, max_new=384):
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors='pt', return_dict=True,
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new, do_sample=False,
            pad_token_id=tokenizer.eos_token_id, repetition_penalty=1.05,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
REDE_SENSITIVE = ['SINAN', 'Ligue 180', 'CVV', 'CAPS', 'SAMU', 'Delegacia da Mulher',
                  'Centro de Referência']
ANTIPADRAO_PACIENTE = ['procure um profissional', 'procure um médico', 'procure ajuda médica',
                       'consulte um profissional', 'fique calma', 'não se preocupe']
RE_DOSE = re.compile(r'\b\d+(?:[\.,]\d+)?\s*(mg|mcg|g|ml|UI|comp|cp)\b', re.IGNORECASE)

def heuristicas(resposta: str, sensitive: bool) -> dict:
    low = resposta.lower()
    return {
        'cita_servico_rede':   any(s.lower() in low for s in REDE_SENSITIVE),
        'antipadrao_paciente': any(a in low for a in ANTIPADRAO_PACIENTE),
        'tem_dose_numerica':   bool(RE_DOSE.search(resposta)),
        'len_chars':           len(resposta),
    }

In [ ]:
# Limite para tempo razoável de avaliação (~3-5 min para 30 exemplos)
N_AMOSTRA = min(30, len(ds_test))
import random
random.seed(42)
indices = random.sample(range(len(ds_test)), k=N_AMOSTRA)

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
resultados = []

for i, idx in enumerate(indices, 1):
    ex = ds_test[idx]
    msg_in = ex['messages'][:2]
    gold   = ex['messages'][2]['content']

    resp_base = gerar(base, msg_in)
    resp_ft   = gerar(ft,   msg_in)

    r_base = scorer.score(gold, resp_base)['rougeL'].fmeasure
    r_ft   = scorer.score(gold, resp_ft  )['rougeL'].fmeasure

    h_base = heuristicas(resp_base, ex['sensitive'])
    h_ft   = heuristicas(resp_ft,   ex['sensitive'])

    resultados.append({
        'idx': idx,
        'category':  ex['category'],
        'sensitive': ex['sensitive'],
        'user':      msg_in[1]['content'],
        'gold':      gold,
        'base':      resp_base,
        'ft':        resp_ft,
        'rouge_base': r_base,
        'rouge_ft':   r_ft,
        'heur_base':  h_base,
        'heur_ft':    h_ft,
    })
    print(f'  [{i}/{N_AMOSTRA}] {ex["category"]:<22} '
          f'rouge base={r_base:.3f}  ft={r_ft:.3f}')
print(f'\nConcluído: {len(resultados)} exemplos avaliados.')

In [ ]:
import statistics as stats

def media(xs): return round(stats.mean(xs), 3) if xs else 0.0
def pct(xs):   return round(100 * sum(xs) / len(xs), 1) if xs else 0.0

# Agregados globais
rouge_b = [r['rouge_base'] for r in resultados]
rouge_f = [r['rouge_ft']   for r in resultados]

print('=' * 60)
print(f'ROUGE-L  base={media(rouge_b):.3f}   ft={media(rouge_f):.3f}'
      f'   delta={media(rouge_f)-media(rouge_b):+.3f}')

# Heurísticas
print(f'\nAntipadrão "procure um médico":')
print(f'  base: {pct([r["heur_base"]["antipadrao_paciente"] for r in resultados])}%')
print(f'  ft:   {pct([r["heur_ft"  ]["antipadrao_paciente"] for r in resultados])}%')

sensitives = [r for r in resultados if r['sensitive']]
if sensitives:
    print(f'\nEm exemplos sensitive (n={len(sensitives)}), cita serviço da rede:')
    print(f'  base: {pct([r["heur_base"]["cita_servico_rede"] for r in sensitives])}%')
    print(f'  ft:   {pct([r["heur_ft"  ]["cita_servico_rede"] for r in sensitives])}%')

print(f'\nComprimento médio (chars):')
print(f'  base: {int(media([r["heur_base"]["len_chars"] for r in resultados]))}')
print(f'  ft:   {int(media([r["heur_ft"  ]["len_chars"] for r in resultados]))}')

# Por categoria
print('\nROUGE-L por categoria:')
por_cat = defaultdict(list)
for r in resultados:
    por_cat[r['category']].append(r)
for cat in sorted(por_cat):
    rs = por_cat[cat]
    mb = media([r['rouge_base'] for r in rs])
    mf = media([r['rouge_ft']   for r in rs])
    print(f'  {cat:<25} n={len(rs):<2}  base={mb:.3f}  ft={mf:.3f}  delta={mf-mb:+.3f}')

In [ ]:
# JSON com resultados completos + agregados
report = {
    'adapter_dir': str(ADAPTER_DIR),
    'n_examples':  len(resultados),
    'aggregates': {
        'rouge_l_base': media(rouge_b),
        'rouge_l_ft':   media(rouge_f),
        'antipadrao_base_pct': pct([r['heur_base']['antipadrao_paciente'] for r in resultados]),
        'antipadrao_ft_pct':   pct([r['heur_ft'  ]['antipadrao_paciente'] for r in resultados]),
        'len_chars_base': int(media([r['heur_base']['len_chars'] for r in resultados])),
        'len_chars_ft':   int(media([r['heur_ft'  ]['len_chars'] for r in resultados])),
    },
    'detalhes': resultados,
}
with open(OUT_REPORT, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print('JSON salvo em:', OUT_REPORT)

# Side-by-side em markdown (10 primeiros)
linhas = ['# Avaliação base vs fine-tuned\n', f'Adapter: `{ADAPTER_DIR}`\n']
for r in resultados[:10]:
    sens = ' [SENSITIVE]' if r['sensitive'] else ''
    linhas.append(f'\n---\n\n## [{r["category"]}{sens}]\n')
    linhas.append(f'**USER**: {r["user"]}\n')
    linhas.append(f'\n**BASE** (ROUGE-L {r["rouge_base"]:.3f}):\n\n{r["base"]}\n')
    linhas.append(f'\n**FINE-TUNED** (ROUGE-L {r["rouge_ft"]:.3f}):\n\n{r["ft"]}\n')
    linhas.append(f'\n**GOLD**:\n\n{r["gold"]}\n')
with open(OUT_SIDEBYSIDE, 'w', encoding='utf-8') as f:
    f.write(''.join(linhas))
print('Markdown salvo em:', OUT_SIDEBYSIDE)